# Groq client

#### 1. Dependencies

In [6]:
%pip install --quiet groq pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import os
import time
import pandas as pd
from groq import Groq

#### 2. Configuration

In [ ]:
MODEL_NAME = "llama-3.3-70b-versatile"
MODEL_SHORT = "Llama-3.3-70B-Groq"

def load_api_key(path, name):
    for line in open(path).read().strip().splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise KeyError(f"Klucz '{name}' nie znaleziony w {path}")

API_KEY = load_api_key("./api_key.txt", "groq")

PROMPTS_PATH = "./prompts.tsv"
OUTPUT_DIR = f"./responses_{MODEL_SHORT}"
RESPONSES_PATH = f"{OUTPUT_DIR}/responses.tsv"

POLITE_DELAY = 35.0
MAX_OUTPUT_TOKENS = 8192

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 3. Load prompts

In [9]:
prompts_df = pd.read_csv(PROMPTS_PATH, sep="\t")

try:
    df = pd.read_csv(RESPONSES_PATH, sep="\t")
    done = df["response"].notna().sum()
    print(f"Wznowiono: {done}/{len(df)} wykonanych")
except FileNotFoundError:
    df = prompts_df.copy()
    df["response"] = pd.NA
    print(f"Start od poczatku: {len(df)} promptow do wyslania")

Wznowiono: 5/12 wykonanych


#### 4. Send loop 

In [ ]:
client = Groq(api_key=API_KEY)

MAX_RETRIES = 4
RETRY_BASE_DELAY = 15.0

def is_transient(exc: Exception) -> bool:
    msg = str(exc).upper()
    return any(kw in msg for kw in ("429", "503", "RATE LIMIT", "OVERLOAD", "TRY AGAIN", "UNAVAILABLE", "TIMEOUT"))

def send_one(prompt: str) -> str:
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_OUTPUT_TOKENS,
            )
            return response.choices[0].message.content or ""
        except Exception as exc:
            if is_transient(exc) and attempt < MAX_RETRIES - 1:
                wait = RETRY_BASE_DELAY * (2 ** attempt)
                print(f"    rate limit / przeciazenie, czekam {wait:.0f}s (proba {attempt + 1}/{MAX_RETRIES})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Nieosiagalne")

pending = df[df["response"].isna()].index.tolist()
print(f"Do wyslania: {len(pending)}")
print(f"Szacowany czas: {len(pending) * POLITE_DELAY / 60:.1f} min")

for i in pending:
    item_id = df.at[i, "id"]
    try:
        text = send_one(df.at[i, "prompt"])
        df.at[i, "response"] = text
        print(f"  [{pending.index(i) + 1}/{len(pending)}] {item_id}: {len(text):,} chars")
    except Exception as exc:
        df.at[i, "response"] = pd.NA
        print(f"  [{pending.index(i) + 1}/{len(pending)}] {item_id}: BLAD - {exc}")

    df.to_csv(RESPONSES_PATH, sep="\t", index=False)
    time.sleep(POLITE_DELAY)

print("Gotowe!")

Do wyslania: 7
Szacowany czas: 4.1 min
    rate limit / przeciazenie, czekam 15s (proba 1/4)
    rate limit / przeciazenie, czekam 30s (proba 2/4)
    rate limit / przeciazenie, czekam 60s (proba 3/4)


In [ ]:

df = pd.read_csv(RESPONSES_PATH, sep="\t")


In [ ]:
df

,id,slug,target_lang,target_lang_name,prompt,response
0,zh--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,zh,Chinese,You are a professional translator. Translate t...,我知道你们都有很长的待办事项清单，但我讨厌浪费时间，所以我有一个不做清单。不要在社交媒体上滚...
1,fi--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fi,Finnish,You are a professional translator. Translate t...,"Tiedän, että teillä kaikilla on pitkät tehtävä..."
2,fr--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fr,French,You are a professional translator. Translate t...,Je sais que vous avez tous des listes de tâche...
3,he--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,he,Hebrew,You are a professional translator. Translate t...,אני יודע שלכולכם יש רשימות ארוכות של דברים שצר...
4,ja--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,ja,Japanese,You are a professional translator. Translate t...,あなたたちは誰しも忙しい生活を送っているので、やることのリストを持っていることでしょう。但し...
5,pl--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,pl,Polish,You are a professional translator. Translate t...,NaN
6,zh--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,zh,Chinese,You are a professional translator. Translate t...,NaN
7,fi--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,fi,Finnish,You are a professional translator. Translate t...,NaN
8,fr--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,fr,French,You are a professional translator. Translate t...,NaN
9,he--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,he,Hebrew,You are a professional translator. Translate t...,NaN
